# 03 Linearization

This notebook converts structured intent-slot records into decoder-ready sequences while handling nulls, empty values, missing required slots, and variable-length lists.

In [1]:
from __future__ import annotations
import json
import logging
from pathlib import Path
from typing import Any, Dict, List
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(message)s')
logger = logging.getLogger('linearization')
ROOT = Path.cwd()
schema = json.loads((ROOT / 'schema.json').read_text(encoding='utf-8'))

In [2]:
NULL_TOKEN = '[NULL]'
EMPTY_TOKEN = '[EMPTY]'

def required_slot_map(schema: Dict[str, Any]) -> Dict[str, List[str]]:
    return {intent['name']: [slot['name'] for slot in intent['slots'] if slot['required']] for intent in schema['intents']}

def normalize_value(value: Any) -> str:
    if value is None: return NULL_TOKEN
    if value == '' or value == []: return EMPTY_TOKEN
    if isinstance(value, list): return ' | '.join(normalize_value(item) for item in value) if value else EMPTY_TOKEN
    if isinstance(value, dict): return ' | '.join(f'{key}={normalize_value(val)}' for key, val in sorted(value.items()))
    value = str(value).strip()
    return value if value else EMPTY_TOKEN

def linearize_sample(sample: Dict[str, Any], schema: Dict[str, Any]) -> str:
    slots = sample.get('slots') or {}
    parts = [f"intent : {sample['intent']}"]
    for slot_name in sorted(slots):
        parts.append(f"{slot_name} : {normalize_value(slots[slot_name])}")
    missing = [name for name in required_slot_map(schema).get(sample['intent'], []) if name not in slots]
    if missing:
        parts.append(f"missing_slots : {' | '.join(missing)}")
    return '[BOS] ' + ' <sep> '.join(parts) + ' [EOS]'

def process_split(split_name: str, schema: Dict[str, Any]) -> List[Dict[str, Any]]:
    records = json.loads((ROOT / f'{split_name}.json').read_text(encoding='utf-8'))
    output = []
    for record in records:
        output.append({'sample_id': record['sample_id'], 'intent': record['intent'], 'slots': record['slots'], 'linearized_sequence': linearize_sample(record, schema), 'target_text': record['target_text'], 'edge_case': record['edge_case'], 'edge_case_types': record['edge_case_types']})
    return output

In [3]:
linearized = {split: process_split(split, schema) for split in ['train', 'validation', 'test']}
for split, records in linearized.items():
    (ROOT / f'linearized_{split}.json').write_text(json.dumps(records, indent=2), encoding='utf-8')
    logger.info('Saved %s records: %d', split, len(records))

preview = pd.DataFrame(linearized['train']).head(5)[['sample_id', 'intent', 'linearized_sequence']]
display(preview)

INFO:Saved train records: 280
INFO:Saved validation records: 60
INFO:Saved test records: 60


,sample_id,intent,linearized_sequence
0,SMP0262,return_item,[BOS] intent : return_item <sep> order_id : OR...
1,SMP0339,return_item,[BOS] intent : return_item <sep> order_id : OR...
2,SMP0218,cancel_order,[BOS] intent : cancel_order <sep> cancellation...
3,SMP0231,refund_status,[BOS] intent : refund_status <sep> order_id : ...
4,SMP0041,order_status,[BOS] intent : order_status <sep> order_id : O...


In [4]:
# Run sanity checks on linearized sequences (robust to schema/data mismatches)
for records in linearized.values():
    for record in records:
        seq = record['linearized_sequence']
        assert seq.startswith('[BOS] intent : ')
        assert seq.endswith(' [EOS]')

